# Data cleaning – NYC Airbnb 2019
Goal: apply the cleaning rules proposed in section 14.4 of the exploration notebook (01_data_exploration.ipynb), and save a cleaned listings file and a host table for the regression and clustering. Each step records how many listings it affects. Rows are removed first (rules 1 and 3), then flags and new columns are added, so that later steps only work on listings that remain.

## 1. Setup
Load the tools and the original dataset from our team's GitHub repository (data/raw). The original data is kept unchanged as df_raw, because the host table is built from all original listings (rule 8 in section 14.4 of the exploration notebook). All cleaning is done on a copy, df.

In [65]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import haversine_distances

url = "https://raw.githubusercontent.com/aabouraya/airbnb-nyc-pricing-segments/refs/heads/main/data/raw/AB_NYC_2019.csv"
df_raw = pd.read_csv(url)
df = df_raw.copy()
df.shape

(48895, 16)

## 2. Remove listings priced at USD 0
Rule 1 (section 14.4 of the exploration notebook): the 11 listings priced at USD 0 are removed, as their true price is unknown.

In [66]:
rows_before = len(df)
df = df.loc[df["price"] > 0].copy()
print("Before:", rows_before, "| After:", len(df), "| Removed:", rows_before - len(df))

Before: 48895 | After: 48884 | Removed: 11


**Result:** 11 listings removed. 48,884 listings remain.

## 3. Remove listings with minimum stays over 365 nights
Rule 3 (section 14.4 of the exploration notebook): listings requiring more than 365 nights are removed, as these are implausible for short-term rentals and often look like placeholder values. Month-long stays are kept.

In [67]:
rows_before = len(df)
df = df.loc[df["minimum_nights"] <= 365].copy()
print("Before:", rows_before, "| After:", len(df), "| Removed:", rows_before - len(df))

Before: 48884 | After: 48870 | Removed: 14


**Result:** 14 listings removed. 48,870 listings remain.

## 4. Flag very high prices
Rule 2 (section 14.4 of the exploration notebook): listings priced above USD 1,000 are kept, but marked with a yes/no column, so the regression can test their effect. As this column is derived from price (the target), it is for diagnostics and sensitivity analysis only and must not be used as a model input.

In [68]:
df["price_above_1000"] = df["price"] > 1000
df["price_above_1000"].sum()

np.int64(239)

**Result:** 239 listings are marked as priced above USD 1,000.

## 5. Flag listings with 0 days available
Rule 4 (section 14.4 of the exploration notebook): listings with 0 days available are kept and marked with a yes/no column, as the data cannot show which of them are inactive.

In [69]:
df["zero_availability"] = df["availability_365"] == 0
df["zero_availability"].sum()

np.int64(17530)

**Result:** 17,530 listings are marked as having 0 days available. (The exploration counted 17,533 in the full dataset; 3 of the listings removed in section 2 also had 0 days available.)

## 6. Missing review values
Rule 5 (section 14.4 of the exploration notebook): a yes/no column marks whether a listing has any reviews. Missing values in reviews_per_month are filled with 0, as section 4.1 of the exploration notebook showed that these listings have no reviews.

In [70]:
df["has_reviews"] = df["number_of_reviews"] > 0
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
print("Listings with reviews:", df["has_reviews"].sum())
print("Missing reviews_per_month left:", df["reviews_per_month"].isna().sum())

Listings with reviews: 38827
Missing reviews_per_month left: 0


**Result:** 38,827 listings have at least one review. No missing values remain in reviews_per_month.

## 7. Days since last review
Rules 5 and 7 (section 14.4 of the exploration notebook): last_review is converted from text to a date, and days_since_last_review is added: the number of days between a listing's last review and the latest review in the original dataset (8 July 2019). It stays empty for listings without reviews, which are marked by has_reviews. It is a supporting variable and is not used as a regression input.

In [71]:
df["last_review"] = pd.to_datetime(df["last_review"])
reference_date = pd.to_datetime(df_raw["last_review"]).max()
df["days_since_last_review"] = (reference_date - df["last_review"]).dt.days

print("Reference date:", reference_date.date())
print("Empty values:", df["days_since_last_review"].isna().sum())

Reference date: 2019-07-08
Empty values: 10043


**Result:** last_review is now stored as a date. days_since_last_review is calculated from 8 July 2019 and is empty for the 10,043 listings without reviews.

## 8. Distance to Midtown
Rule 7 (section 14.4 of the exploration notebook): dist_midtown_km is added, the straight-line distance in kilometres from each listing to Times Square (latitude 40.758896, longitude -73.985130), used as the centre of Midtown, as in section 12.4 of the exploration notebook.

In [72]:
coords = np.radians(df[["latitude", "longitude"]].to_numpy())
center = np.radians([[40.758896, -73.985130]])
df["dist_midtown_km"] = haversine_distances(coords, center)[:, 0] * 6371

df["dist_midtown_km"].describe()

,dist_midtown_km
count,48870.000000
mean,7.135964
std,4.446434
min,0.032125
25%,3.857566
50%,6.404815
75%,9.466834
max,35.996613


**Result:** dist_midtown_km has been added for all 48,870 listings, ranging from 0.03 km to 36 km (median 6.4 km), consistent with the exploration.

## 9. Drop host_name
Rule 6 (section 14.4 of the exploration notebook): host_name is removed, as it contains personal data and is not needed for the analysis. name (the listing title) is kept for reference only.

In [73]:
df = df.drop(columns=["host_name"])
df.columns.tolist()

['id',
 'name',
 'host_id',
 'neighbourhood_group',
 'neighbourhood',
 'latitude',
 'longitude',
 'room_type',
 'price',
 'minimum_nights',
 'number_of_reviews',
 'last_review',
 'reviews_per_month',
 'calculated_host_listings_count',
 'availability_365',
 'price_above_1000',
 'zero_availability',
 'has_reviews',
 'days_since_last_review',
 'dist_midtown_km']

**Result:** host_name has been removed. The cleaned dataset has 20 columns: the 15 remaining original columns and 5 new ones (price_above_1000, zero_availability, has_reviews, days_since_last_review, dist_midtown_km).

## 10. Final check
A last check of the size of the cleaned dataset and of the remaining missing values.

In [74]:
print(df.shape)
df.isna().sum()

(48870, 20)


,0
id,0
name,15
host_id,0
neighbourhood_group,0
neighbourhood,0
latitude,0
longitude,0
room_type,0
price,0
minimum_nights,0


**Result:** The cleaned dataset has 48,870 listings and 20 columns. Missing values remain only in name (15 listing titles, not used in the models) and in last_review and days_since_last_review (10,043 each, the listings without reviews, marked by has_reviews).

## 11. Host table
Rule 8 (section 14.4 of the exploration notebook): one row per host, built from all 48,895 original listings (df_raw), independently of the price-related cleaning. Columns: number of listings (the original calculated_host_listings_count), average availability, share of listings with 0 days available, share of entire homes, median minimum stay and number of neighbourhood groups. For the median minimum stay, values above 365 nights are treated as missing; hosts without any valid value keep an empty median, to be handled during clustering preprocessing. Price and review columns are not included.

In [75]:
listings = df_raw.copy()
listings["is_entire_home"] = listings["room_type"] == "Entire home/apt"
listings["is_zero_availability"] = listings["availability_365"] == 0
listings["min_nights_valid"] = listings["minimum_nights"].where(listings["minimum_nights"] <= 365)

listings[["is_entire_home", "is_zero_availability"]].sum()

,0
is_entire_home,25409
is_zero_availability,17533


**Result:** Of the 48,895 original listings, 25,409 are entire homes and 17,533 have 0 days available.

In [76]:
hosts = listings.groupby("host_id").agg(
    n_listings=("calculated_host_listings_count", "first"),
    avg_availability=("availability_365", "mean"),
    share_zero_availability=("is_zero_availability", "mean"),
    share_entire_home=("is_entire_home", "mean"),
    median_min_nights=("min_nights_valid", "median"),
    n_neighbourhood_groups=("neighbourhood_group", "nunique"),
).reset_index()

hosts.head()

,host_id,n_listings,avg_availability,share_zero_availability,share_entire_home,median_min_nights,n_neighbourhood_groups
0,2438,1,0.000000,1.0,1.0,45.0,1
1,2571,1,23.000000,0.0,1.0,9.0,1
2,2787,6,235.333333,0.0,0.0,1.0,1
3,2845,2,360.000000,0.0,0.5,1.0,1
4,2868,1,221.000000,0.0,1.0,29.0,1


In [77]:
print("Shape:", hosts.shape)
print("Sum of n_listings:", hosts["n_listings"].sum())
print("Listings with minimum stay over 365:", listings["min_nights_valid"].isna().sum())
print("Hosts without a valid median minimum stay:", hosts["median_min_nights"].isna().sum())
hosts.describe().round(2)

Shape: (37457, 7)
Sum of n_listings: 48895
Listings with minimum stay over 365: 14
Hosts without a valid median minimum stay: 13


,host_id,n_listings,avg_availability,share_zero_availability,share_entire_home,median_min_nights,n_neighbourhood_groups
count,3.745700e+04,37457.00,37457.00,37457.00,37457.00,37444.00,37457.00
mean,6.144549e+07,1.31,88.29,0.43,0.55,5.68,1.00
std,7.511750e+07,2.76,118.35,0.49,0.49,15.96,0.06
min,2.438000e+03,1.00,0.00,0.00,0.00,1.00,1.00
25%,7.050126e+06,1.00,0.00,0.00,0.00,1.50,1.00
50%,2.679050e+07,1.00,16.00,0.00,1.00,2.50,1.00
75%,8.817184e+07,1.00,164.00,1.00,1.00,4.00,1.00
max,2.743213e+08,327.00,365.00,1.00,1.00,365.00,4.00


**Result:** The host table has 37,457 hosts (one row each) and 7 columns. The listing counts add up to 48,895, so every original listing is counted exactly once. 14 listings had minimum stays above 365 nights and were treated as missing; as a result, 13 hosts have no valid median minimum stay, which is left empty for clustering preprocessing. All values lie within plausible ranges. n_neighbourhood_groups varies very little: almost all hosts operate in a single neighbourhood group (maximum 4), so it may add little to the clustering.

## 12. Validation checks
Automated checks that the cleaned files are as expected. If any check fails, the notebook stops with an error before the files are saved, so later changes to the cleaning cannot silently produce wrong data.

In [78]:
# Cleaned listings
assert len(df) == 48870
assert (df["price"] > 0).all()
assert (df["minimum_nights"] <= 365).all()
assert df["id"].is_unique
assert "host_name" not in df.columns
assert df["reviews_per_month"].notna().all()

# Host table
assert len(hosts) == 37457
assert hosts["host_id"].is_unique
assert hosts["n_listings"].sum() == len(df_raw)
assert (hosts.set_index("host_id")["n_listings"] == listings.groupby("host_id").size()).all()

print("All checks passed.")

All checks passed.


## 13. Save the cleaned files
The cleaned listings (listings_clean.csv) and the host table (hosts.csv) are saved as CSV files and downloaded, then reloaded to verify that they are complete. The downloaded files are uploaded manually to the repository (data/processed), so all parts work from the same version. Section 14 explains how to use them.

In [79]:
from google.colab import files

df.to_csv("listings_clean.csv", index=False)
hosts.to_csv("hosts.csv", index=False)

files.download("listings_clean.csv")
files.download("hosts.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [80]:
check_listings = pd.read_csv("listings_clean.csv", parse_dates=["last_review"])
check_hosts = pd.read_csv("hosts.csv")

assert check_listings.shape == df.shape
assert check_hosts.shape == hosts.shape
assert check_listings["host_id"].isin(check_hosts["host_id"]).all()

print("Both saved files passed verification.")

Both saved files passed verification.


## 14. Using the cleaned files
**Important for modelling:** the files contain identifiers, the regression target, diagnostic flags and supporting review variables, which must not be confused with model inputs. In particular, price_above_1000 must never be used to predict price, and review variables are excluded from the regression, as agreed by the team.

**listings_clean.csv**

| Column | Use |
|---|---|
| price | Regression target |
| neighbourhood_group, latitude, longitude, dist_midtown_km, room_type, minimum_nights, calculated_host_listings_count, availability_365, zero_availability | Possible regression inputs |
| neighbourhood | Describing results; possible input only if tested (many small neighbourhoods) |
| id | Identifies listings; not a model input |
| host_id | Grouping for the cross-validation split, and linking listings to host types; not a model input |
| name | Reference only |
| price_above_1000 | Diagnostics and sensitivity analysis only; never a model input (derived from price) |
| number_of_reviews, reviews_per_month, last_review, days_since_last_review, has_reviews | Supporting analysis (e.g. comparing review activity with price deviations); not regression inputs |

**hosts.csv**

| Column | Use |
|---|---|
| host_id | Identifies hosts and links them to listings; not a clustering input |
| n_listings, avg_availability, share_zero_availability, share_entire_home, median_min_nights, n_neighbourhood_groups | Possible clustering inputs |

**Notes for the modelling**
- **Loading:** a CSV file stores dates as text. When loading listings_clean.csv, last_review needs to be converted to a date again, e.g. with pd.read_csv(..., parse_dates=["last_review"]).
- **Linking the files:** listings and hosts are linked through host_id. Every listing in listings_clean.csv has its host in hosts.csv, but a few hosts in hosts.csv have no listings left in listings_clean.csv (e.g. hosts whose only listing was priced at USD 0). This is expected, as the host table is built from all original listings.
- **Listing counts:** calculated_host_listings_count and n_listings count each host's original listings, including any removed during cleaning, so they may differ slightly from a count of listings in listings_clean.csv.
- **Regression:** zero_availability and availability_365 overlap (all zero_availability listings have availability_365 = 0), so using both may add little.
- **Clustering:** avg_availability and share_zero_availability both describe availability; using both may give availability too much weight in K-means. It may be worth comparing results with and without one of them.
- **Clustering:** n_neighbourhood_groups varies very little (almost all hosts operate in a single neighbourhood group), so it may add little.
- **Clustering:** 13 hosts have no valid median_min_nights (see section 11); these need to be filled in or excluded before K-means.
- **Clustering:** n_listings is very skewed (1 to 327), so it should be log-transformed and all features scaled before K-means.

## 15. Summary of the cleaning steps

**Cleaned listings (listings_clean.csv)**

| Step | Rule | Effect |
|---|---|---|
| Starting point | – | 48,895 listings, 16 columns |
| Remove listings priced at USD 0 | 1 | 11 listings removed |
| Remove minimum stays over 365 nights | 3 | 14 listings removed |
| Flag prices above USD 1,000 | 2 | 239 listings marked (price_above_1000) |
| Flag 0 days available | 4 | 17,530 listings marked (zero_availability) |
| Mark listings with reviews | 5 | 38,827 listings with reviews (has_reviews) |
| Fill missing reviews_per_month with 0 | 5 | 10,043 values filled |
| Convert last_review and add days since last review | 5, 7 | days_since_last_review added; empty for 10,043 listings without reviews |
| Add distance to Midtown | 7 | dist_midtown_km added for all listings |
| Drop host_name | 6 | Column removed |
| **Result** | – | **48,870 listings, 20 columns** |

**Host table (hosts.csv)**

| Step | Rule | Effect |
|---|---|---|
| Starting point | 8 | All 48,895 original listings, independently of the price-related cleaning |
| Minimum stays over 365 nights treated as missing | 8 | 14 listings; 13 hosts without a valid median minimum stay |
| **Result** | – | **37,457 hosts, 7 columns** |

All rules refer to section 14.4 of the exploration notebook.